### 1. Preparação do Ambiente e Importação de Bibliotecas
Nesta célula inicial, configuramos o ambiente de trabalho carregando as bibliotecas essenciais para a Ciência de Dados e Machine Learning:

* **Pandas e Numpy**: Fundamentais para a manipulação de matrizes e tabelas de dados biomecânicos.
* **Scikit-Learn**: Fornece as ferramentas de divisão de dados (`train_test_split`), normalização (`StandardScaler`) e as métricas de validação.
* **XGBoost e Random Forest**: Os algoritmos de inteligência artificial que serão treinados para identificar o risco de entorse.
* **OS e Time**: Bibliotecas de sistema para gestão de ficheiros e medição da eficiência (tempo de execução) dos modelos.

In [2]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS (VERSÃO FINAL)
# Responsabilidade: Carregar ferramentas de Manipulação, ML, Deep Learning e Gráficos.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time

# 1. Visualização de Dados (Para os Gráficos de Importância e Matrizes)
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Machine Learning Tradicional (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, recall_score, f1_score, 
                             confusion_matrix, classification_report)
from sklearn.ensemble import RandomForestClassifier

# 3. Modelos de Elite (XGBoost)
try:
    from xgboost import XGBClassifier
    print("✅ XGBoost: Carregado.")
except ImportError:
    print("⚠️ XGBoost não instalado. Use: !pip install xgboost")

# 4. Deep Learning (PyTorch)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    print("✅ PyTorch: Carregado.")
except ImportError:
    print("⚠️ PyTorch não instalado. Use: !pip install torch")

# Configurações Estéticas para o Artigo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("🚀 Todas as bibliotecas foram carregadas com sucesso!")

✅ XGBoost: Carregado.
✅ PyTorch: Carregado.
🚀 Todas as bibliotecas foram carregadas com sucesso!


### 2. Carregamento e Consolidação dos Dados Brutos
Nesta etapa, o sistema acede à diretoria externa `Data` para importar os conjuntos de dados biomecânicos provenientes dos três artigos científicos base. 

* **Caminhos Relativos**: Utilizamos a biblioteca `os` para garantir que o código seja portátil entre diferentes sistemas operativos (Windows/Mac/Linux).
* **Tratamento de Exceções**: Implementámos um bloco `try-except` para capturar erros de ficheiro não encontrado, garantindo que o programa não falhe abruptamente caso a pasta `Data` esteja ausente ou mal posicionada.
* **Validação Inicial**: O sistema confirma o sucesso do carregamento exibindo o volume total de amostras prontas para processamento.

In [3]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO DE DADOS
# Responsabilidade: Aceder à pasta 'Data' e carregar os datasets localmente.
# ==============================================================================
print("--- FASE 1: CARREGAMENTO ---")

# Caminho relativo (funciona em qualquer PC com a pasta Data ao lado do Notebook)
BASE_PATH = "../Data" 

try:
    # Construindo os caminhos de forma robusta
    df1 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 1.csv"), sep=',')
    df2 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 2.csv"), sep=',')
    df3 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 3.csv"), sep=',')
    
    print("✅ Arquivos carregados com sucesso!")
    print(f"Total de amostras: {len(df1) + len(df2) + len(df3)}")
    load_successful = True
    
except FileNotFoundError as e:
    print(f"❌ ERRO: Não encontrei a pasta ou os ficheiros em: {BASE_PATH}")
    load_successful = False

--- FASE 1: CARREGAMENTO ---
✅ Arquivos carregados com sucesso!
Total de amostras: 181


### 3. Engenharia de Features e Pré-processamento de Dados
Esta fase é crucial para garantir a qualidade das previsões do modelo **PlaySafe4All**. O objetivo é transformar dados heterogéneos numa matriz numérica padronizada.

* **Consolidação e Seleção**: Unimos os diferentes datasets e filtramos apenas as variáveis biomecânicas e de exposição (Match/Training Time) que têm relevância clínica para a entorse.
* **Tratamento de Dados Omissos**: Utilizamos a **Mediana** para preencher valores em falta, garantindo que o modelo não seja enviesado por valores extremos (outliers).
* **Codificação e Binarização**: O alvo (*Target*) é convertido para um formato binário (0 para saudável, 1 para entorse), facilitando a interpretação pelos algoritmos de classificação.
* **Divisão Estratificada**: Dividimos os dados em Treino e Teste mantendo a proporção original de lesões (`stratify=y`), o que evita que o modelo aprenda apenas sobre atletas saudáveis.
* **Normalização (StandardScaler)**: Reescalamos todas as variáveis para que tenham média 0 e desvio padrão 1. Isto impede que variáveis com números grandes (como o tempo em minutos) dominem variáveis com números pequenos (como razões de força).

In [5]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO (SIMPLIFICADA)
# Responsabilidade: Transformar dados brutos em matrizes de treino/teste.
# ==============================================================================

print("\n--- FASE 2: PREPARAÇÃO DE DADOS ---")

# 1. Unir as bases e remover colunas inúteis (como ID/Número)
df = pd.concat([df1, df2, df3], ignore_index=True)
if 'Numero' in df.columns: df.drop(columns=['Numero'], inplace=True)

# 2. Seleção de colunas relevantes e Alvo
TARGET = 'Entorse'
cols = [TARGET, 'T0_T1_Match_Time_exposure', 'T0_T1_Training_Time_exposure', 
        'T0SRTMax', 'T0TTestMin', 'T0SJmMax', 'T0Veli', 'T0RazaoAADto', 'T0RazaoAAEsq']

# Mantém apenas o que existe nos ficheiros e preenche vazios com a mediana
df = df[[c for c in cols if c in df.columns]].copy()
df = df.fillna(df.median(numeric_only=True))

# 3. Definição de X e Y (Ajustando Y para 0 e 1 se necessário)
y = df[TARGET].apply(lambda x: 1 if x > 1 else 0) # Garante formato binário 0/1
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)

# 4. Divisão e Escalonamento (Standardization)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"✅ Dados Prontos! Treino: {X_train_scaled.shape} | Teste: {X_test_scaled.shape}")


--- FASE 2: PREPARAÇÃO DE DADOS ---
✅ Dados Prontos! Treino: (45, 56) | Teste: (136, 56)


### 4. Construção e Treino do Modelo (XGBoost)
Nesta etapa, implementámos o algoritmo XGBoost, uma técnica de aprendizagem supervisionada de elite baseada em Gradient Boosted Decision Trees. O modelo foi escolhido pela sua elevada eficiência em conjuntos de dados tabulares e pela sua capacidade de lidar com relações não-lineares entre métricas físicas e o risco de lesão.

Lógica de Funcionamento:

Aprendizagem Sequencial: Ao contrário de modelos como o Random Forest (onde as árvores são independentes), o XGBoost utiliza o princípio de boosting. O algoritmo cria árvores de forma sequencial, onde cada nova árvore é desenhada especificamente para corrigir os erros de classificação (resíduos) cometidos pelas árvores anteriores.

Configuração dos Hiperparâmetros (Otimização):
Para garantir a robustez da previsão e evitar o overfitting (viciação do modelo), foram definidos os seguintes parâmetros:

n_estimators=100: Estabelece a criação de 100 árvores sequenciais para a composição do modelo final.

max_depth=3: Limita a profundidade máxima de cada árvore. Uma profundidade baixa (3 níveis) força o modelo a aprender padrões globais e simples, impedindo que ele "decore" ruídos específicos da amostra de 181 registos.

learning_rate=0.05: Também conhecido como shrinkage, este parâmetro controla o ritmo de atualização dos pesos. Uma taxa de 0.05 garante uma convergência mais lenta e estável, permitindo que o modelo refine a aprendizagem sem saltar por cima da solução ideal.

eval_metric='logloss': O objetivo matemático do modelo é a minimização da perda logarítmica (logloss), garantindo que a probabilidade atribuída a cada previsão (Saudável vs. Entorse) seja o mais próxima possível da realidade clínica observada.

In [16]:
# ==============================================================================
# CÉLULA 4: MODELO XGBOOST (O COMPETIDOR DE ELITE)
# ==============================================================================
print("\n--- FASE 3: XGBOOST (GRADIENT BOOSTING) ---")

# Verificação de segurança: Se a Célula 3 não foi rodada, o código avisa
if 'X_train_scaled' not in locals():
    print("❌ ERRO: Precisas de correr a CÉLULA 3 primeiro para criar os dados!")
else:
    # 1. Configuração do Modelo
    # Learning_rate baixa ajuda a não "viciar" nos teus 181 dados
    modelo_xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    # 2. Treino
    print("🚀 Treinando o XGBoost (Aprendizagem Sequencial)...")
    modelo_xgb.fit(X_train_scaled, y_train)

    # 3. Predição
    y_pred_xgb = modelo_xgb.predict(X_test_scaled)
    acc_xgb = accuracy_score(y_test, y_pred_xgb)

    print(f"✅ XGBoost Concluído! Precisão: {acc_xgb*100:.2f}%")


    print("\n--- MATRIZ DE CONFUSÃO (XGBOOST) ---")
    print(confusion_matrix(y_test, y_pred_xgb))


--- FASE 7: XGBOOST (GRADIENT BOOSTING) ---
🚀 Treinando o XGBoost (Aprendizagem Sequencial)...
✅ XGBoost Concluído! Precisão: 69.12%

--- MATRIZ DE CONFUSÃO (XGBOOST) ---
[[44 17]
 [25 50]]


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:40:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### 5. Avaliação de Desempenho e Segurança Biometral
Nesta fase final, o modelo XGBoost foi submetido a um rigoroso protocolo de teste utilizando dados que o algoritmo não encontrou durante a fase de treino. Este procedimento garante a validade externa do modelo e a sua capacidade de generalização para cenários reais de treino e competição.

Métricas de Performance Operacional
Para quantificar a eficácia do sistema PlaySafe4All, utilizámos quatro indicadores fundamentais:

Accuracy (Precisão Global): Mede a taxa de acerto total do modelo em todas as previsões (Saudável e Lesão).

Recall (Sensibilidade): Esta é a nossa métrica mais crítica. Indica a percentagem de lesões reais que o modelo conseguiu prever corretamente. No contexto desportivo, um Recall elevado é vital para garantir que atletas em risco sejam identificados antes do evento lesivo.

F1-Score: Representa a média harmónica entre a precisão e a sensibilidade, servindo como um indicador de estabilidade e equilíbrio do algoritmo, especialmente importante em bases de dados onde as lesões são eventos menos frequentes que a normalidade.

Eficiência Temporal: O tempo de execução foi monitorizado em milissegundos (ms) para assegurar que o sistema é capaz de fornecer alertas instantâneos, permitindo a tomada de decisão imediata por parte da equipa técnica à beira do campo.

Matriz de Confusão e Segurança do Atleta
A validação clínica é centrada na Matriz de Confusão, onde monitorizamos especificamente:

Verdadeiros Positivos: Lesões corretamente antecipadas.

Falsos Negativos (FN): Atletas em risco que o sistema classificou erroneamente como "Saudáveis".

Nota Crítica: No projeto PlaySafe4All, o limiar de decisão (threshold) foi ajustado para 0.4. Esta escolha deliberada prioriza a redução dos Falsos Negativos em detrimento de um ligeiro aumento nos falsos alarmes, garantindo que a integridade física do atleta seja a prioridade máxima do sistema.

In [14]:
# ==============================================================================
# CÉLULA 5: AVALIAÇÃO FINAL (XGBOOST)
# Responsabilidade: Extração de métricas puras para tabelas de resultados.
# ==============================================================================
import time
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

print("\n--- FASE 4: MÉTRICAS DE PERFORMANCE (XGBoost) ---")

if 'modelo_xgb' not in locals():
    print("❌ ERRO: O modelo 'modelo_xgb' não foi treinado!")
else:
    start_time = time.perf_counter()

    # 1. Previsões e Probabilidades
    y_probs = modelo_xgb.predict_proba(X_test_scaled)[:, 1]
    # Threshold de 0.4 para priorizar a detecção de lesões (Segurança do Atleta)
    y_pred = (y_probs > 0.4).astype(int) 

    # 2. Cálculos de Performance
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    execution_time_ms = (time.perf_counter() - start_time) * 1000

    # 3. Exibição Direta para Cópia (Resultados do Artigo)
    print("-" * 40)
    print(f"✅ Precisão Global (Accuracy): {accuracy*100:.2f}%")
    print(f"✅ Recall (Sensibilidade):    {recall*100:.2f}%")
    print(f"✅ F1-Score:                 {f1:.4f}")
    print(f"✅ Tempo de Resposta:        {execution_time_ms:.2f} ms")
    print("-" * 40)

    print("\n--- MATRIZ DE CONFUSÃO ---")
    print(f"Saudáveis previstos corretamente: {cm[0,0]}")
    print(f"Falsos Alarmes (Saudável -> Risco): {cm[0,1]}")
    print(f"Lesões não detetadas (FN):        {cm[1,0]}")
    print(f"Lesões detetadas corretamente:    {cm[1,1]}")
    
    

    # 4. Verificação de Validação
    if cm[1, 0] <= 2:
        print("\n🏆 CONCLUSÃO: Modelo validado com alta capacidade preditiva de lesões.")
    else:
        print(f"\n⚠️ NOTA: O modelo falhou em {cm[1,0]} casos. Verifique o equilíbrio dos dados.")


--- FASE 4: MÉTRICAS DE PERFORMANCE (XGBoost) ---
----------------------------------------
✅ Precisão Global (Accuracy): 66.18%
✅ Recall (Sensibilidade):    76.00%
✅ F1-Score:                 0.7125
✅ Tempo de Resposta:        19.19 ms
----------------------------------------

--- MATRIZ DE CONFUSÃO ---
Saudáveis previstos corretamente: 33
Falsos Alarmes (Saudável -> Risco): 28
Lesões não detetadas (FN):        18
Lesões detetadas corretamente:    57

⚠️ NOTA: O modelo falhou em 18 casos. Verifique o equilíbrio dos dados.
